In [ ]:
from pathlib import Path
import itertools
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sym
from configobj import ConfigObj

REPO = Path.cwd().resolve()
if REPO.name == "Notebooks":
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import HiCOLA.Frontend.expression_builder as eb
import HiCOLA.Frontend.numerical_solver as ns
from HiCOLA.Frontend.read_parameters import read_in_parameters

exec(eb.declare_symbols())
ODEINT_PARAMETER_SYMBOLS = [E, phiprime, omegar, omegam]


In [ ]:
PLANCK_INI = REPO / "HiCOLA/Parameter_files/Frontend_parameter_files/examples/Planck18_cosmological_parameters.ini"
TEMPLATE_INI = REPO / "HiCOLA/Parameter_files/Frontend_parameter_files/nDGP_scan/nDGP_scan.ini"
GENERATED_INI_DIR = REPO / "HiCOLA/Parameter_files/Frontend_parameter_files/nDGP_scan/generated"
RESULTS_CSV = REPO / "HiCOLA/Output/nDGP_scan/scan_results.csv"

F_PHI_VALUES = np.linspace(0.0, 1.0, 6)
E_DS_VALUES = np.linspace(0.80, 0.94, 8)

EPSILON = 1e-6
OMEGA_M_CRIT = 0.99
RUN_FRONTEND_COMMAND = False


In [ ]:
def lcdm_density_today(planck_ini):
    cfg = ConfigObj(str(planck_ini))
    h = cfg.as_float("h")

    if cfg["Omega_m0"] == "None":
        omega_m0 = (cfg.as_float("Omega_b0h2") + cfg.as_float("Omega_c0h2")) / h**2
    else:
        omega_m0 = cfg.as_float("Omega_m0")

    if cfg["Omega_r0"] == "None":
        omega_r0 = cfg.as_float("Omega_r0h2") / h**2
    else:
        omega_r0 = cfg.as_float("Omega_r0")

    omega_lambda0 = 1.0 - omega_m0 - omega_r0
    return omega_r0, omega_m0, omega_lambda0


def rc_H0_from_rule(f_phi, e_ds, omega_lambda0):
    alpha = 1.0 - ((1.0 - f_phi) * omega_lambda0) / e_ds**2
    if not np.isfinite(alpha) or alpha >= 0.0:
        return np.nan, alpha

    rc_H0 = 1.0 / (e_ds * (-alpha)**0.25)
    if not np.isfinite(rc_H0) or rc_H0 <= 0.0:
        return np.nan, alpha

    return rc_H0, alpha


OMEGA_R0_LCDM, OMEGA_M0_LCDM, OMEGA_LAMBDA0_LCDM = lcdm_density_today(PLANCK_INI)
OMEGA_R0_LCDM, OMEGA_M0_LCDM, OMEGA_LAMBDA0_LCDM


In [ ]:
def scan_label(f_phi, e_ds):
    return f"nDGP_scan_f{f_phi:.3f}_EdS{e_ds:.3f}".replace(".", "p")


def scan_ini_path(label):
    return GENERATED_INI_DIR / f"{label}.ini"


def write_scan_ini(f_phi, e_ds, rc_H0, label):
    ini_path = scan_ini_path(label)

    cfg = ConfigObj(str(TEMPLATE_INI))
    cfg.filename = str(ini_path)
    cfg["model_name"] = label
    cfg["output_directory"] = str(REPO / "HiCOLA/Output/nDGP_scan")
    cfg["f_phi"] = f"{f_phi:.16g}"
    cfg["E_dS"] = f"{e_ds:.16g}"
    cfg["Hubble_dS"] = f"{e_ds:.16g}"
    cfg["Hubble0"] = f"{1.0 / e_ds:.16g}"
    cfg["rc_H0_parameter"] = f"{rc_H0:.16g}"

    GENERATED_INI_DIR.mkdir(parents=True, exist_ok=True)
    cfg.write()
    return ini_path


def prepare_solver_input(ini_path):
    read_out = read_in_parameters(str(ini_path), str(PLANCK_INI))
    read_out.update({"odeint_parameter_symbols": ODEINT_PARAMETER_SYMBOLS})
    read_out.update(
        eb.create_Horndeski(
            read_out["K"],
            read_out["G3"],
            read_out["G4"],
            read_out["symbol_list"],
            read_out["mass_ratio_list"],
        )
    )
    return read_out


def run_frontend_command(ini_path):
    cmd = [
        sys.executable,
        "-m",
        "HiCOLA.Frontend.generate_simulation_input",
        str(ini_path),
        str(PLANCK_INI),
    ]
    return subprocess.run(cmd, cwd=REPO, check=True, capture_output=True, text=True)


In [ ]:
def classify_solution(background):
    z = 1.0 / background["a"] - 1.0
    densities = {
        "omega_m": np.asarray(background["omega_m"], dtype=float),
        "omega_r": np.asarray(background["omega_r"], dtype=float),
        "omega_l": np.asarray(background["omega_l"], dtype=float),
        "omega_phi": np.asarray(background["omega_phi"], dtype=float),
    }

    density_violation = any(
        (~np.isfinite(values)).any()
        or (values < -EPSILON).any()
        or (values > 1.0 + EPSILON).any()
        for values in densities.values()
    )

    omega_de = np.asarray(background["omega_DE"], dtype=float)
    today_index = np.nanargmin(np.abs(z))
    high_z = z > 1.0
    early_de_violation = np.nanmax(omega_de[high_z]) > omega_de[today_index] + EPSILON

    max_omega_m = np.nanmax(densities["omega_m"])
    omega_m_crit_violation = max_omega_m < OMEGA_M_CRIT

    return {
        "density_violation": bool(density_violation),
        "early_de_violation": bool(early_de_violation),
        "omega_m_crit_violation": bool(omega_m_crit_violation),
        "max_omega_m": float(max_omega_m),
        "max_omega_de_z_gt_1": float(np.nanmax(omega_de[high_z])),
        "omega_de_z0": float(omega_de[today_index]),
    }


def run_scan():
    rows = []
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

    for f_phi, e_ds in itertools.product(F_PHI_VALUES, E_DS_VALUES):
        f_phi = float(f_phi)
        e_ds = float(e_ds)
        label = scan_label(f_phi, e_ds)
        ini_path = scan_ini_path(label)
        rc_H0, alpha = rc_H0_from_rule(f_phi, e_ds, OMEGA_LAMBDA0_LCDM)
        invalid_rc_H0 = not np.isfinite(rc_H0)
        row = {
            "label": label,
            "f_phi": f_phi,
            "E_dS": e_ds,
            "alpha": float(alpha),
            "rc_H0": float(rc_H0),
            "ini_path": str(ini_path),
            "ini_generated": False,
            "invalid_rc_H0": bool(invalid_rc_H0),
            "solver_error": "",
        }

        if invalid_rc_H0:
            row.update(
                {
                    "density_violation": False,
                    "early_de_violation": False,
                    "omega_m_crit_violation": False,
                    "max_omega_m": np.nan,
                    "max_omega_de_z_gt_1": np.nan,
                    "omega_de_z0": np.nan,
                    "solver_error": "invalid rc_H0",
                    "viable": False,
                    "status": "invalid rc_H0",
                }
            )
            rows.append(row)
            continue

        try:
            ini_path = write_scan_ini(f_phi, e_ds, rc_H0, label)
            row["ini_generated"] = True

            if RUN_FRONTEND_COMMAND:
                run_frontend_command(ini_path)

            solver_input = prepare_solver_input(ini_path)
            background = ns.run_solver(solver_input)
            row.update(classify_solution(background))
        except Exception as exc:
            row.update(
                {
                    "density_violation": True,
                    "early_de_violation": True,
                    "omega_m_crit_violation": True,
                    "max_omega_m": np.nan,
                    "max_omega_de_z_gt_1": np.nan,
                    "omega_de_z0": np.nan,
                    "solver_error": repr(exc),
                }
            )

        row["viable"] = not (
            row["density_violation"]
            or row["early_de_violation"]
            or row["omega_m_crit_violation"]
        )
        row["status"] = "viable" if row["viable"] else "not viable"
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    return df


In [ ]:
def plot_scan(df):
    fig, ax = plt.subplots(figsize=(6.8, 4.2), dpi=180)

    viable = df["viable"].astype(bool)
    invalid_rc_H0 = df["invalid_rc_H0"].astype(bool)
    not_viable = ~viable & ~invalid_rc_H0
    ax.scatter(
        df.loc[not_viable, "f_phi"],
        df.loc[not_viable, "E_dS"],
        s=72,
        marker="x",
        linewidths=1.8,
        color="0.45",
        label="not viable",
    )
    ax.scatter(
        df.loc[invalid_rc_H0, "f_phi"],
        df.loc[invalid_rc_H0, "E_dS"],
        s=96,
        marker="P",
        color="#8e44ad",
        edgecolor="black",
        linewidth=0.6,
        label=r"invalid $r_c H_0$",
    )
    ax.scatter(
        df.loc[viable, "f_phi"],
        df.loc[viable, "E_dS"],
        s=96,
        marker="o",
        color="#2a9d55",
        edgecolor="black",
        linewidth=0.6,
        label="viable",
    )

    overlays = [
        ("density_violation", "density bounds", "#d62728", "s"),
        ("early_de_violation", "early DE", "#ff9f1c", "D"),
        ("omega_m_crit_violation", "max Omega_m", "#1f77b4", "^"),
    ]
    for column, label, color, marker in overlays:
        mask = df[column].astype(bool) & ~invalid_rc_H0
        ax.scatter(
            df.loc[mask, "f_phi"],
            df.loc[mask, "E_dS"],
            s=44,
            marker=marker,
            facecolors="none",
            edgecolors=color,
            linewidths=1.2,
            label=label,
        )

    ax.set_xlabel(r"$f_\phi$")
    ax.set_ylabel(r"$E_{\rm dS}$")
    ax.set_xlim(min(F_PHI_VALUES) - 0.08, max(F_PHI_VALUES) + 0.08)
    ax.set_ylim(min(E_DS_VALUES) - 0.015, max(E_DS_VALUES) + 0.015)
    ax.grid(True, color="0.88", linewidth=0.7)
    ax.tick_params(direction="in", top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    ax.legend(
        frameon=False,
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.16),
        ncol=3,
        columnspacing=1.2,
        handletextpad=0.4,
    )
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    return fig, ax


scan_results = run_scan()
display(scan_results)
plot_scan(scan_results)
